## 03 - Optimization-Driven Experimental Design

This notebook explains how to turn surrogate-model predictions into experimental decisions. We focus on proposing feasible polymer formulations that push target properties while honoring chemistry constraints and articulating the usual trade-offs (strength vs. elongation vs. thermal performance). The workflow highlights three building blocks for a scientific ML portfolio:

- constrained candidate generation
- uncertainty-aware selection (exploration vs. exploitation)
- single- and multi-objective views (scalar scores + Pareto front)


In [ ]:
# Notebook bootstrap: ensure imports resolve from the project root
import sys
from pathlib import Path
import warnings

warnings.filterwarnings(
    'ignore',
    message='X has feature names, but DecisionTreeRegressor was fitted without feature names',
)


project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

from src.data_generation import generate_synthetic_formulations, simulate_lab_results

sns.set_theme(style="whitegrid")

SEED = 7

input_cols = [
    "polymer_A",
    "polymer_B",
    "plasticizer",
    "filler",
    "stabilizer",
    "temperature",
    "curing_time",
    "pressure",
]

target_cols = ["tensile_strength", "elongation", "thermal_resistance"]

# Synthetic "ground truth" to emulate the world (not part of the actual system)
df_full = generate_synthetic_formulations(500, seed=SEED, noise_level=1.0, sum_tolerance=0.25)

# Consulting scenario: only 40 experiments available
df_train = df_full.sample(40, random_state=42).copy()

X_train = df_train[input_cols]
y_train = df_train[target_cols]

(df_full.shape, df_train.shape)

In [ ]:
# Non-linear surrogate trained with scarce data
# Also leverage tree-to-tree dispersion as a lightweight uncertainty proxy.
surrogate = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=800,
        random_state=SEED,
        min_samples_leaf=2,
        bootstrap=True,
    )
)

surrogate.fit(X_train, y_train)

"trained"

### 1) Constrained candidate generation

The optimizer operates only inside the physically realizable region. We first build a large candidate pool by enforcing:

- variable-level bounds (wt% and process variables)
- the ~100 wt% mass balance constraint

This is a guided random search: every point we evaluate already satisfies the laboratory heuristics before the surrogate scores it, and the same pool feeds the downstream optimization variants.


In [ ]:
N_CANDIDATES = 20000

# IMPORTANT: here we only use the generator to obtain feasible inputs.
# In a real lab these targets would not exist; they come from the surrogate.
df_candidates = generate_synthetic_formulations(
    N_CANDIDATES,
    seed=123,
    noise_level=1.0,
    sum_tolerance=0.25,
)

X_cand = df_candidates[input_cols]

# (Predictions and their uncertainty are computed in the next cell.)

df_candidates.head()

### 2) Uncertainty-aware next experiment (explore vs exploit)

The Random Forest surrogate provides a lightweight standard deviation via tree dispersion. We convert it into an **Upper Confidence Bound (UCB)** rule:

\( \text{UCB} = \mu + \kappa \sigma \)

- mu captures expected performance (exploitation).
- sigma rewards poorly explored regions (exploration).
- kappa balances both, which is valuable in low-data settings.

This layer showcases uncertainty-aware selection without relying on a full Bayesian Optimization stack yet.


### 2.1) Quick read of the suggested formulation

For each selected candidate we expose its composition and processing recipe. That makes it easy to sanity-check whether the recommendation aligns with materials intuition (e.g., high filler to boost 	ensile_strength, moderate plasticizer to retain stiffness, temperature within the optimal curing window).


**Technical interpretation**

- Elevated 
iller usually accompanies strength-oriented objectives; a mid-range value avoids hurting elongation too much.
- Moderate/low plasticizer helps protect 	ensile_strength without losing ductility entirely.
- 	emperature and curing_time near the sweet spot reinforce that the proposal follows processing know-how.

These notes behave like an engineering checklist before green-lighting the next laboratory run.


In [ ]:
def rf_mean_std(mo_rf: MultiOutputRegressor, X: pd.DataFrame):
    """Return (mean, std) per target using per-tree predictions."""
    means = []
    stds = []
    for est in mo_rf.estimators_:
        # shape: (n_trees, n_rows)
        per_tree = np.stack([t.predict(X) for t in est.estimators_], axis=0)
        means.append(per_tree.mean(axis=0))
        stds.append(per_tree.std(axis=0))

    mean = np.stack(means, axis=1)  # (n_rows, n_targets)
    std = np.stack(stds, axis=1)
    return mean, std


mean_pred, std_pred = rf_mean_std(surrogate, X_cand)

df_pred = df_candidates[input_cols].copy()
for i, c in enumerate(target_cols):
    df_pred[c] = mean_pred[:, i]
    df_pred[c + "_std"] = std_pred[:, i]

# Single-objective: maximizar resistencia esperada (media)
best_exploit = df_pred.sort_values("tensile_strength", ascending=False).head(1)

# Next experiment (UCB): maximizar media + kappa*std
KAPPA = 1.0

df_pred["tensile_strength_ucb"] = df_pred["tensile_strength"] + KAPPA * df_pred["tensile_strength_std"]
best_next = df_pred.sort_values("tensile_strength_ucb", ascending=False).head(1)

print("Best exploit (highest mean strength)")
display(best_exploit)
print("\nNext experiment (UCB: mean + kappa*std)")
display(best_next)
# Detalle del candidato seleccionado
best = best_exploit.iloc[0]

best_inputs = best[[
    "polymer_A",
    "polymer_B",
    "plasticizer",
    "filler",
    "stabilizer",
    "temperature",
    "curing_time",
    "pressure",
]]

best_outputs = best[[
    "tensile_strength",
    "elongation",
    "thermal_resistance",
    "tensile_strength_std",
]]

display(pd.DataFrame({"value": best_inputs}).T)
display(pd.DataFrame({"value": best_outputs}).T)


In [ ]:
def train_surrogate(df_train: pd.DataFrame, seed: int = 0):
    X = df_train[input_cols]
    y = df_train[target_cols]

    model = MultiOutputRegressor(
        RandomForestRegressor(
            n_estimators=800,
            random_state=seed,
            min_samples_leaf=2,
            bootstrap=True,
        )
    )
    model.fit(X, y)
    return model


def propose_next_experiment(model, *, seed: int, kappa: float = 1.0, n_candidates: int = 15000):
    df_cand = generate_synthetic_formulations(n_candidates, seed=seed, noise_level=1.0, sum_tolerance=0.25)
    Xc = df_cand[input_cols]

    mu, sig = rf_mean_std(model, Xc)
    mu_strength = mu[:, 0]
    sig_strength = sig[:, 0]

    ucb = mu_strength + kappa * sig_strength
    idx = int(np.argmax(ucb))

    row = df_cand.iloc[[idx]][input_cols].copy()
    meta = {
        "mu_strength": float(mu_strength[idx]),
        "sigma_strength": float(sig_strength[idx]),
        "ucb": float(ucb[idx]),
    }
    return row, meta


def run_iterative_loop(df_start: pd.DataFrame, *, n_steps: int = 5, kappa: float = 1.0, seed: int = 0):
    df_hist = df_start.copy().reset_index(drop=True)
    log = []

    for step in range(n_steps):
        model = train_surrogate(df_hist, seed=seed + step)
        x_next, meta = propose_next_experiment(model, seed=1000 + step, kappa=kappa)

        # "Lab": measure targets for that candidate (simulated here)
        y_meas = simulate_lab_results(x_next, seed=2000 + step, noise_level=1.0)

        new_row = pd.concat([x_next.reset_index(drop=True), y_meas.reset_index(drop=True)], axis=1)
        df_hist = pd.concat([df_hist, new_row], ignore_index=True)

        log.append({"step": step, **meta, **y_meas.iloc[0].to_dict()})

    return df_hist, pd.DataFrame(log)


df_after, loop_log = run_iterative_loop(df_train, n_steps=5, kappa=1.0, seed=SEED)

loop_log

### 3) Multi-objective: weighted score + Pareto front

Many materials problems require maximizing several properties at once. Here we combine two complementary readings:

1. **Weighted score**: normalize the objectives and apply tunable weights (e.g., strength > elongation > thermal resistance).
2. **Pareto front**: extract non-dominated candidates to visualize trade-offs and pick according to context.

This section makes it explicit that the pipeline supports multi-objective optimization and delivers the Pareto frontier needed to justify choices to stakeholders.


In [ ]:
def minmax_norm(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    return (s - s.min()) / (s.max() - s.min() + 1e-12)


df_score = df_pred.copy()

# Objectives (all maximized) with built-in trade-offs
for c in target_cols:
    df_score[c + "_n"] = minmax_norm(df_score[c])

# Weights (adjust them to fit the case narrative)
W_STRENGTH = 0.45
W_ELONG = 0.35
W_THERM = 0.20

df_score["score"] = (
    W_STRENGTH * df_score["tensile_strength_n"]
    + W_ELONG * df_score["elongation_n"]
    + W_THERM * df_score["thermal_resistance_n"]
)

best_multi = df_score.sort_values("score", ascending=False).head(5)

best_multi[["score"] + input_cols + target_cols].head()

In [ ]:
def pareto_front(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Return non-dominated points for a maximization problem."""
    X = df[cols].to_numpy(dtype=float)
    n = X.shape[0]
    is_dominated = np.zeros(n, dtype=bool)

    for i in range(n):
        if is_dominated[i]:
            continue
        # j dominates i if all >= and at least one >
        ge = np.all(X >= X[i], axis=1)
        gt = np.any(X > X[i], axis=1)
        dominates_i = ge & gt
        # if any point dominates i, mark i
        if np.any(dominates_i):
            is_dominated[i] = True

    return df.loc[~is_dominated].copy()


# For speed, approximate the Pareto front on a random subset of candidates
subset = df_pred.sample(3000, random_state=0)
front = pareto_front(subset, target_cols)

print("subset:", subset.shape, "pareto_front:", front.shape)
front.head()

In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(data=subset, x="elongation", y="tensile_strength", alpha=0.25, label="candidates")
sns.scatterplot(data=front, x="elongation", y="tensile_strength", alpha=0.9, label="pareto front")
plt.title("Trade-off view (Pareto front approximation)")
plt.legend()
plt.show()

### Final Conclusion

The full workflow - physics-respecting synthetic data + surrogate modeling + constraint-aware optimization - shows how a portfolio project can document an iterative experimental design loop:

- Synthetic data keeps the story scientifically honest and reproducible.
- Surrogate models quantify how trustworthy each prediction is in the low-data regime.
- Optimization layers add uncertainty handling and multi-objective visibility to prioritize experiments without marketing fluff.

Bottom line: this pipeline accelerates learning and narrows the lab queue without pretending to replace real validation.


### 2.2) Exploration sweep (kappa = 0, 1, 2)

We contrast three configurations for the UCB rule:

- kappa = 0.0: pure exploitation (surrogate mean only).
- kappa = 1.0: balanced trade-off between expected performance and uncertainty.
- kappa = 2.0: exploration-heavy, useful at the beginning of a campaign.

Watching how the recommendation shifts helps communicate the impact of the exploration parameter inside the experimental loop.


In [ ]:
# Compare the next experiment for different exploration levels (kappa)
kappas = [0.0, 1.0, 2.0]

rows = []
for k in kappas:
    tmp = df_pred.copy()
    tmp["tensile_strength_ucb"] = tmp["tensile_strength"] + k * tmp["tensile_strength_std"]
    best_k = tmp.sort_values("tensile_strength_ucb", ascending=False).iloc[0]

    rows.append({
        "kappa": k,
        "tensile_strength_mu": best_k["tensile_strength"],
        "tensile_strength_sigma": best_k["tensile_strength_std"],
        "ucb": best_k["tensile_strength_ucb"],
        "filler": best_k["filler"],
        "plasticizer": best_k["plasticizer"],
        "temperature": best_k["temperature"],
    })

kappa_summary = pd.DataFrame(rows)
kappa_summary

### 5.1) Iterative loop summary by kappa

Running multiple iterations of the 	rain -> propose -> simulate -> retrain loop reveals:

- Low kappa: converges quickly to promising regions but may leave blind spots.
- High kappa: sacrifices some expected performance to reduce uncertainty and surface alternative formulations.

Documenting this dynamic proves that the workflow supports deliberate exploration vs. exploitation decisions.


In [ ]:
# Comparativa del loop iterativo para distintos kappa
loop_rows = []
for k in [0.0, 1.0, 2.0]:
    _, log_k = run_iterative_loop(df_train, n_steps=10, kappa=k, seed=SEED)
    last = log_k.iloc[-1]
    loop_rows.append({
        "kappa": k,
        "last_step": int(last["step"]),
        "last_mu_strength": last["mu_strength"],
        "last_sigma_strength": last["sigma_strength"],
        "last_measured_strength": last["tensile_strength"],
    })

loop_summary = pd.DataFrame(loop_rows)
loop_summary

### 4) Next upgrade: Bayesian / multi-objective BO

The constrained random pool works as a baseline, but the natural upgrade is to plug in Bayesian Optimization or multi-objective evolutionary search:

- Probabilistic surrogates (Gaussian Processes, calibrated ensembles) to model uncertainty more faithfully.
- Acquisition functions such as Expected Improvement or Hypervolume Improvement to explore the space with higher sample efficiency.

This roadmap makes it clear how to scale the project without changing its scientific core.


### 5) Iterative experimental design loop

The documented loop is:

1. Start from historical data (~40 measurements) or the equivalent synthetic dataset.
2. Train the surrogate and estimate its uncertainty.
3. Generate feasible candidates and prioritize them with the chosen strategy (kappa/UCB, weighted score, Pareto front).
4. Run or simulate the experiment and capture the new measurements.
5. Update the dataset, retrain, and repeat.

This makes the case that you can keep a reproducible loop inside a portfolio project without relying on proprietary lab data.


In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(loop_log["step"], loop_log["mu_strength"], label="predicted mean")
plt.fill_between(
    loop_log["step"],
    loop_log["mu_strength"] - loop_log["sigma_strength"],
    loop_log["mu_strength"] + loop_log["sigma_strength"],
    alpha=0.2,
    label="+/-1 std (uncertainty)",
)
plt.plot(loop_log["step"], loop_log["tensile_strength"], marker="o", label="measured (simulated lab)")
plt.title("Iterative loop: predicted vs measured strength")
plt.xlabel("Iteration")
plt.ylabel("tensile_strength")
plt.legend()
plt.tight_layout()
plt.show()